알츠하이머 분류를 위한 최종 정형 데이터인 hippo_features_clean_icv.csv 파일 필요함

추가적으로 10개의 피처명 일치하는지 확인하고 실행시키기


In [1]:
# 1. 필수 라이브러리 설치 (Colab에서 한 번 실행)
!pip install pandas xgboost scikit-learn

# 2. 라이브러리 임포트
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, recall_score, precision_score
from xgboost import XGBClassifier

# 3. Google Drive 마운트 (데이터 파일 접근)
from google.colab import drive
drive.mount('/content/drive')

# ----------------------------------------------------
# 4. 데이터 파일 로드 (학교 컴퓨터에서 가져온 CSV 파일 경로 설정)
# 예시 경로: drive/MyDrive/Memora_data/hippo_features_clean_icv.csv
FILE_PATH = '/content/drive/MyDrive/Memora_data/hippo_features_clean_icv.csv'

try:
    df_raw = pd.read_csv(FILE_PATH)
    print("✅ 데이터 로드 성공. 상위 5개 행:")
    print(df_raw.head())
    print(f"\n총 데이터 개수: {len(df_raw)}")
except FileNotFoundError:
    print("❌ 오류: 파일을 찾을 수 없습니다. FILE_PATH를 확인해주세요.")

# ----------------------------------------------------
# 5. 데이터 준비 (필요한 컬럼만 선택)
# 5. 데이터 클리닝 및 준비

LABEL_COL = 'label'
ALL_COLS_TO_CHECK = FEATURES_10 + [LABEL_COL]

# 1. 결측치 확인 및 제거
initial_count = len(df_raw)
df_clean = df_raw.dropna(subset=ALL_COLS_TO_CHECK)
removed_count = initial_count - len(df_clean)

if removed_count > 0:
    print(f"⚠️ 경고: {removed_count}개의 행(샘플)에 결측치가 있어 제거되었습니다. (총 {len(df_clean)}개 사용)")

# 2. 데이터 타입을 숫자로 강제 변환 (오류 방지)
for col in ALL_COLS_TO_CHECK:
    # 'coerce' 옵션으로 숫자로 변환할 수 없는 값은 NaN으로 만듭니다.
    # (단, 위에서 NaN은 이미 제거했으므로 남은 오류를 방지하는 용도입니다.)
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

df_raw = df_clean # 이제 클린된 데이터프레임을 사용

print("\n✅ 최종 데이터 준비 완료. 컬럼 타입 확인:")
print(df_raw[ALL_COLS_TO_CHECK].dtypes)

# [cite_start]레이블 (정답) 컬럼: CN=0, AD=1 [cite: 30, 97]
# [cite_start]상세설계서에 따르면 BL(Baseline Data), SC(Screening Data)를 사용하여 라벨링함 [cite: 20]
LABEL_COL = 'label'

# [cite_start]최종 10개 피처 [cite: 92]
FEATURES_10 = [
    'left_hipp_vol_mm3', 'right_hipp_vol_mm3', 'total_hipp_vol_mm3',
    'asymmetry_index',
    'left_hipp_vol_icv_norm', 'right_hipp_vol_icv_norm', 'total_hipp_vol_icv_norm',
    'AGE', 'APOE4', 'SEX_FEMALE' # 임상/유전 위험 인자
]

# MRI 기반 7개 피처 (임상/유전 제외)
FEATURES_7 = FEATURES_10[:7]

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/operations/check.py", line 101, in check_install_conflicts
    package_set, _ = create_package_set_from_installed()
              

KeyboardInterrupt: 

In [ ]:
## --- 데이터 분할 및 학습 함수 정의 ---

def split_data(df, features, target_col, split_ratio, random_state=42):
    """
    데이터셋을 지정된 비율(80:20 또는 70:10:20)로 분할합니다.
    """
    X = df[features]
    y = df[target_col]

    # [cite_start]Stratified split: CN/AD 비율을 유지하며 분할 [cite: 98]
    if split_ratio == '80:20':
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, stratify=y, random_state=random_state)
        X_val, y_val = None, None

    elif split_ratio == '70:10:20':
        # 1차 분할: (70+10)% vs 20% (Test)
        X_train_val, X_test, y_train_val, y_test = train_test_split(
            X, y, test_size=0.2, stratify=y, random_state=random_state)

        # 2차 분할: Train/Validation (70% vs 10%)
        X_train, X_val, y_train, y_val = train_test_split(
            X_train_val, y_train_val, test_size=0.125, stratify=y_train_val, random_state=random_state)
            # 0.25 * 0.5 = 0.125 (0.8 * 0.125 = 0.1)

    print(f"\n[Split: {split_ratio}] Train={len(X_train)}, Val={len(X_val) if X_val is not None else 0}, Test={len(X_test)}")
    return X_train, X_val, X_test, y_train, y_val, y_test


def evaluate_model(model, X_test, y_test, title=""):
    """
    모델을 평가하고 AUC, Accuracy, Recall, Precision을 계산합니다.
    (최적 임곗값 대신 0.5 기본값 사용 - 최적화 로직은 복잡하여 제외)
    """
    # 예측 확률 (y_prob)
    y_prob = model.predict_proba(X_test)[:, 1]

    # [cite_start]기본 임곗값 0.5 적용 [cite: 99]
    y_pred = (y_prob >= 0.5).astype(int)

    # [cite_start]AUC 계산 [cite: 99]
    auc = roc_auc_score(y_test, y_prob)

    # [cite_start]지표 계산 [cite: 99]
    accuracy = accuracy_score(y_test, y_pred) # 정확도 (Accuracy)
    recall = recall_score(y_test, y_pred) # 민감도 (Sensitivity/Recall)
    specificity = recall_score(y_test, y_pred, pos_label=0) # 특이도 (Specificity)
    precision = precision_score(y_test, y_pred)

    print(f"\n--- {title} 결과 ---")
    print(f"✅ AUC (Area Under Curve): {auc:.4f} (목표: ~0.90)")
    print(f"✅ 정확도 (Accuracy): {accuracy:.4f} (목표: ~0.85)")
    print(f"   민감도 (Sensitivity/Recall): {recall:.4f}")
    print(f"   특이도 (Specificity): {specificity:.4f}")
    print(f"   정밀도 (Precision): {precision:.4f}")

    return auc, accuracy

In [ ]:
# 모델 설정 (기본 XGBoost 파라미터)
xgb_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'use_label_encoder': False,
    'random_state': 42
}

results = []

# =========================================================================
# 🔴 시나리오 1: 피처 10개 (임상+MRI) & 80:20 분할
# =========================================================================
print("\n" + "="*80)
print("🔴 시나리오 1: 피처 10개 (임상+MRI) & 학습/테스트 80:20")
print("="*80)

# 데이터 분할
X_train1, _, X_test1, y_train1, _, y_test1 = split_data(
    df_raw, FEATURES_10, LABEL_COL, '80:20')

# 모델 학습
model1 = XGBClassifier(**xgb_params)
model1.fit(X_train1, y_train1)

# 모델 평가
auc1, acc1 = evaluate_model(model1, X_test1, y_test1,
                            title="시나리오 1 (10개 피처, 80:20) 테스트 결과")
results.append({'Scenario': '10-F, 80:20', 'AUC': auc1, 'Accuracy': acc1})


# =========================================================================
# 🔴 시나리오 2: 피처 10개 (임상+MRI) & 70:10:20 분할
# =========================================================================
print("\n" + "="*80)
print("🔴 시나리오 2: 피처 10개 (임상+MRI) & 학습/검증/테스트 70:10:20")
print("="*80)

# 데이터 분할
X_train2, X_val2, X_test2, y_train2, y_val2, y_test2 = split_data(
    df_raw, FEATURES_10, LABEL_COL, '70:10:20')

# 모델 학습 (검증 데이터셋은 Early Stopping 등에 활용 가능하지만 여기서는 단순 학습)
model2 = XGBClassifier(**xgb_params)
model2.fit(X_train2, y_train2)

# 모델 평가
auc2, acc2 = evaluate_model(model2, X_test2, y_test2,
                            title="시나리오 2 (10개 피처, 70:10:20) 테스트 결과")
results.append({'Scenario': '10-F, 70:10:20', 'AUC': auc2, 'Accuracy': acc2})


# =========================================================================
# 🔴 시나리오 3: 피처 7개 (MRI 전용) & 80:20 분할
# =========================================================================
print("\n" + "="*80)
print("🔴 시나리오 3: 피처 7개 (MRI 전용) & 학습/테스트 80:20")
print("="*80)

# 데이터 분할
X_train3, _, X_test3, y_train3, _, y_test3 = split_data(
    df_raw, FEATURES_7, LABEL_COL, '80:20')

# 모델 학습
model3 = XGBClassifier(**xgb_params)
model3.fit(X_train3, y_train3)

# 모델 평가
auc3, acc3 = evaluate_model(model3, X_test3, y_test3,
                            title="시나리오 3 (7개 피처, 80:20) 테스트 결과")
results.append({'Scenario': '7-F, 80:20', 'AUC': auc3, 'Accuracy': acc3})


# =========================================================================
# 🔴 시나리오 4: 피처 7개 (MRI 전용) & 70:10:20 분할
# =========================================================================
print("\n" + "="*80)
print("🔴 시나리오 4: 피처 7개 (MRI 전용) & 학습/검증/테스트 70:10:20")
print("="*80)

# 데이터 분할
X_train4, X_val4, X_test4, y_train4, y_val4, y_test4 = split_data(
    df_raw, FEATURES_7, LABEL_COL, '70:10:20')

# 모델 학습
model4 = XGBClassifier(**xgb_params)
model4.fit(X_train4, y_train4)

# 모델 평가
auc4, acc4 = evaluate_model(model4, X_test4, y_test4,
                            title="시나리오 4 (7개 피처, 70:10:20) 테스트 결과")
results.append({'Scenario': '7-F, 70:10:20', 'AUC': auc4, 'Accuracy': acc4})


# =========================================================================
# 🌟 최종 결과 요약
# =========================================================================
print("\n" + "="*80)
print("🌟🌟 최종 검증 결과 요약 🌟🌟")
print("="*80)
results_df = pd.DataFrame(results)
print(results_df)